# Simulation Multi-Agents du Trafic — Stratégies de Coordination aux Intersections

**Preuve de technologie & banc d'essai scientifique**

> Ce notebook est le rapport reproductible du simulateur. Les sections
> rédigées (état de l'art, modèles, architecture) sont complètes ; les cellules
> de figures lisent les CSV produits par le moteur C++ (`MetricsCollector`,
> `ExperimentRunner`/`simbench`). Tant que vous n'avez pas déposé vos CSV dans
> `analysis/data/`, les figures s'affichent à partir de **données synthétiques
> étiquetées** afin que le notebook s'exécute de bout en bout.

**Résumé.** Nous présentons un simulateur de trafic *microscopique* multi-agents
dont chaque véhicule suit une dynamique longitudinale continue (Intelligent
Driver Model) et une dynamique latérale de changement de voie (MOBIL), sur des
trajectoires curvilignes (repère de Frenet). Plusieurs paradigmes de résolution
de conflit aux intersections sont implémentés et comparés : feux tricolores,
priorité fixe, négociation pair-à-pair (P2P/VANET), réservation centralisée
(AIM), évitement réciproque (ORCA) et peloton virtuel. Nous mesurons débit,
délai, sécurité (TTC) et confort (jerk) par balayage Monte-Carlo *headless*, et
nous étudions le point d'inflexion où une coordination décentralisée se dégrade
sous forte densité — une illustration du **paradoxe de Braess**.

## 1. Introduction & problématique

Le trafic routier est un **système multi-agents (SMA)** par excellence : des
décisions *locales* (accélérer, freiner, céder, doubler) produisent des
phénomènes *globaux* (ondes de congestion, blocages, débit de saturation).
L'objectif de ce travail est double :

1. **Reproduire** des comportements de conduite réalistes à partir de modèles
   physiques reconnus, plutôt que de règles *ad hoc* déterministes.
2. **Évaluer quantitativement** différentes stratégies de coordination aux
   intersections dans un environnement fortement contraint (jusqu'à >1000
   agents simultanés), pour identifier *quand* et *pourquoi* une approche
   décentralisée surpasse — ou s'effondre face à — une régulation classique.

La question centrale : *à partir de quelle densité une négociation locale
(P2P) cesse-t-elle d'être avantageuse par rapport à une régulation par feux ou
à une réservation centralisée ?*

## 2. État de l'art & modèles implémentés

### 2.1. Cinématique longitudinale — Intelligent Driver Model (IDM)

L'IDM (Treiber, Hennecke & Helbing, 2000) est un modèle de *car-following*
continu, sans collision et calibré sur des observations empiriques.
L'accélération d'un véhicule de vitesse $v$, dont le leader est à une distance
nette (pare-chocs à pare-chocs) $s$ et de différence de vitesse
$\Delta v = v - v_{\text{lead}}$, vaut :

$$ a_{\text{IDM}} = a_{\max}\left[\,1 - \left(\frac{v}{v_0}\right)^{\delta}
   - \left(\frac{s^{*}(v,\Delta v)}{s}\right)^{2}\right] $$

$$ s^{*}(v,\Delta v) = s_0 + \max\!\left(0,\; v\,T + \frac{v\,\Delta v}
   {2\sqrt{a_{\max}\,b}}\right) $$

où $v_0$ est la vitesse désirée, $T$ le temps inter-véhiculaire de sécurité,
$s_0$ l'écart minimal à l'arrêt, $a_{\max}$ l'accélération maximale et $b$ la
décélération confortable. Le terme libre $[1-(v/v_0)^\delta]$ tire vers $v_0$ ;
le terme d'interaction $-(s^*/s)^2$ freine d'autant plus que l'on s'approche du
leader. Implémentation : `src/core/behavior/IdmModel.{hpp,cpp}`.

> **Subtilité — point d'arrêt fixe.** Pour un « leader virtuel » immobile (ligne
> de stop, feu rouge), le terme de *time-headway* $v\,T$ n'a pas de sens (il
> sert à rester $T$ secondes *derrière un mobile*). Le `LeaderInfo` porte un
> drapeau `stopTarget` qui désactive ce terme — sinon le véhicule freine 2–3×
> trop tôt puis flue interminablement vers la ligne.

### 2.2. Cinématique longitudinale curviligne — repère de Frenet

Les trajectoires (`Lane`) sont paramétrées par leur **abscisse curviligne** $s$
(coordonnée de Frenet le long de l'arc), et non en $(x,y)$ cartésien. La
sélection du leader projette chaque véhicule voisin sur *ma* trajectoire :

- abscisse du pied de projection $s_{\text{proj}}$ → **gap mesuré le long du
  tracé** (et non à vol d'oiseau) : correct même en virage ;
- écart latéral signé $d_\perp$ → **filtre same-lane** : un véhicule en
  contre-sens ou sur une voie perpendiculaire projette un $|d_\perp|$ grand et
  n'est *jamais* retenu comme leader.

C'est la correction du bug d'« accrochage » décrit §5.1. Implémentation :
`Lane::project()` + `Perception::scan()`.

### 2.3. Comportement latéral — MOBIL

Le changement de voie / dépassement suit la logique **MOBIL** (Kesting, Treiber
& Helbing, 2007) : *Minimizing Overall Braking Induced by Lane changes*. Deux
critères :

- **Incitation** : le changement n'est intéressant que si mon accélération y
  gagne, pondérée par un facteur de *politesse* $p$ tenant compte de la gêne
  infligée aux suiveurs.
- **Sécurité (critère absolu)** : la décélération imposée au nouveau suiveur
  (et, sur route bidirectionnelle, au trafic en sens inverse) ne doit pas
  dépasser un seuil $b_{\text{safe}}$. En pratique on calcule la décélération
  requise $a_{\text{req}} = v_{\text{rel}}^2 / (2\,d)$ pour éviter le frontal ;
  si $a_{\text{req}} > b_{\text{safe}}$ **ou** le TTC est trop court, le
  dépassement est refusé/avorté. Implémentation : `Vehicle::tryStartOvertake`,
  `isOncomingLaneFree`, `isReturnSlotFree`.

### 2.4. Stratégies de négociation aux intersections

| Stratégie | Paradigme | Référence clé | Module |
|-----------|-----------|---------------|--------|
| Feux tricolores | Régulation temporelle fixe | — (baseline) | `TrafficLightPolicy` |
| Priorité à droite / fixe | Règles de priorité statiques | code de la route | `PriorityRightPolicy`, `FixedPriorityPolicy` |
| STOP / YIELD | Gap-acceptance | — | `StopPolicy` |
| **P2P (VANET)** | Négociation décentralisée, *claims* asynchrones, dominance par règles asymétriques | Au & Stone ; VanMiddlesworth, Kuhlman & Stone (2008) | `P2PPolicy` |
| **AIM** | Réservation centralisée de tuiles spatio-temporelles | Dresner & Stone (2008), *Autonomous Intersection Management* | `AimPolicy` |
| **ORCA** | Évitement réciproque continu (vitesses optimales) | van den Berg, Guy, Lin & Manocha (2011) | `OrcaPolicy` |
| **Peloton virtuel** | Projection 1D + paires meneur/suiveur (zipping) | inspiré CACC / platooning | `PlatooningPolicy` |
| Rond-point | Insertion cédant à l'anneau | — | `RoundaboutPolicy` |

### 2.5. Interblocage (gridlock), spillback & « keep clear »

Dans un réseau dense, une régulation purement locale dégénère en **blocage
mutuel** : des véhicules s'engagent alors que leur *sortie* est déjà pleine, se
figent sur l'aire de conflit et verrouillent les flux croisés (*spillback*).
Deux familles de remèdes existent : (i) un **gestionnaire centralisé** (AIM)
qui réserve l'espace-temps et n'autorise l'entrée que si la sortie est
garantie ; (ii) une **règle locale « keep clear »** — ne pas s'engager si l'on
ne peut pas dégager — combinée à un **bris de cycle** déterministe (ordre total
par identifiant) pour casser les blocages rotatifs à $N$ véhicules. Nous
implémentons (ii) (§5.2).

### 2.6. Paradoxe de Braess

Ajouter de la capacité (ou, ici, *libéraliser* la coordination) peut **dégrader**
la performance globale du réseau au-delà d'un seuil : c'est le paradoxe de
Braess (1968). Nos mesures §7 montrent que le P2P, optimal à faible densité,
finit par se comporter comme un carrefour « 4 STOP » saturé — une manifestation
directe de ce paradoxe.

**Références.**
- M. Treiber, A. Hennecke, D. Helbing (2000). *Congested traffic states in
  empirical observations and microscopic simulations.* Phys. Rev. E **62**, 1805.
- A. Kesting, M. Treiber, D. Helbing (2007). *General lane-changing model MOBIL
  for car-following models.* Transportation Research Record **1999**, 86–94.
- K. Dresner, P. Stone (2008). *A multiagent approach to autonomous intersection
  management.* JAIR **31**, 591–656.
- T.-C. Au, P. Stone — protocoles de réservation/négociation décentralisés ;
  M. VanMiddlesworth, K. Kuhlman, P. Stone (2008), *Replacing the stop sign:
  unmanaged intersection control for autonomous vehicles* (AAMAS).
- J. van den Berg, S. J. Guy, M. Lin, D. Manocha (2011). *Reciprocal n-body
  collision avoidance (ORCA).* Robotics Research, 3–19.
- D. Braess (1968). *Über ein Paradoxon aus der Verkehrsplanung.*

## 3. Architecture logicielle & optimisations

Le moteur est **découplé du rendu** : toute la logique vit dans `core/` et
`sim/` (sans dépendance SFML), le rendu dans `render/`. Cela permet d'exécuter
des milliers de pas en *headless* (Monte-Carlo) à la vitesse du CPU.

**Pipeline strict en deux phases** (déterministe, parallélisable) :

1. `computeDecision(agents, world)` — *lecture seule* : perception (Frenet) +
   choix de la vitesse désirée + leader réel/virtuel + IDM → `pendingAccel`.
   Aucune mutation visible des autres agents.
2. `integrate(dt)` — intégration cinématique (Euler, `dt` fixe = 1/60 s) :
   mise à jour de $s$, position et cap depuis la `Lane`.

Cette séparation garantit que *tous* les agents décident vis-à-vis du **même**
état global (pas de pollution séquentielle), ce qui rend la phase 1
**embarrassingly parallel**.

**Multithreading & partitionnement.** `ParallelDecisions` + `ThreadPool`
répartissent la phase 1 sur tous les cœurs ; les négociations locales d'une
intersection A sont calculées indépendamment de l'intersection B. Le coût
naïf des interactions est $O(N^2)$ ; la perception borne ses tests au voisinage
(rejet par distance avant projection) et la décision est *thread-safe* (chaque
agent ne lit que sa propre `Lane` et les positions des autres).

**Banc de mesure.** `MetricsCollector` échantillonne à chaque pas débit, délai,
vitesse, TTC, arrêts et jerk — 100 % observationnel (n'utilise que l'interface
publique `IAgent`). `ExperimentRunner` orchestre le balayage Monte-Carlo
(`stratégie × densité × runs`) et exporte un CSV. `simbench` mesure la
performance moteur (secondes simulées / seconde réelle).

## 4. Modules du moteur (carte du code)

| Couche | Fichiers | Rôle |
|--------|----------|------|
| Agents | `core/agent/{Vehicle,Car,Truck}.cpp` | état, pipeline décision/intégration, dépassement, panne |
| Car-following | `core/behavior/IdmModel.cpp` | IDM (+ variante CACC dans `Vehicle`) |
| Perception | `core/perception/Perception.cpp` | scan + **leader par Frenet** |
| Trajectoire | `core/world/Lane.cpp` | abscisse curviligne, `project()` |
| Intersections | `core/intersection/*Policy.cpp` | feux, priorité, STOP, P2P, AIM, ORCA, peloton, rond-point |
| Pathfinding | `core/pathfinding/AStarPlanner.hpp` | A* sur la grille |
| Métriques | `core/metrics/MetricsCollector.cpp` | débit/délai/TTC/jerk + export CSV/JSON |
| Monte-Carlo | `sim/ExperimentRunner.cpp`, `sim/McLiveSession.cpp` | balayage headless / visuel |
| Parallélisme | `sim/ParallelDecisions.cpp`, `sim/ThreadPool.cpp` | phase 1 multi-cœurs |
| Rendu | `render/SfmlRenderer.cpp`, `render/Camera.cpp` | dessin + caméra (zoom) |

## 5. Bugs corrigés (avant de passer à l'échelle)

### 5.1. Faux « SUIT » — leader fantôme en virage / à l'intersection

**Symptôme.** Un véhicule fraîchement sorti d'une intersection affichait le
drapeau *SUIT* (suivi de leader) alors que le seul véhicule présent roulait sur
la **voie en sens inverse**.

**Cause.** La sélection du leader était purement **angulaire** : cône étroit
(8°) + tolérance de cap (45°). En courbe, le vrai leader sort du cône et un
véhicule mal placé y entre ; aucune notion de « même voie ».

**Correction (Frenet).** `Lane::project()` projette chaque voisin sur ma
trajectoire ; n'est retenu comme leader qu'un véhicule (i) **devant** moi le
long du tracé ($s_{\text{proj}} > s$), (ii) **dans le couloir** de ma voie
($|d_\perp| \le 22$ px — la voie opposée est à ~50 px), (iii) de cap aligné.
Indépendant de tout cône → robuste en virage.

> **À INTÉGRER :** capture *avant* (faux SUIT sur la voie opposée) — votre
> `Capture d'écran 2026-05-27 135843.png` — puis une capture *après* correction.

### 5.2. Gridlock (ville XXXXL) — « keep clear »

**Symptôme.** Files ininterrompues bloquées aux carrefours ; le réseau dense
fige (spillback).

**Correction.** (a) Règle **« keep clear »** : avant de s'engager, le véhicule
vérifie par projection de Frenet que la **sortie** du carrefour (juste après,
sur sa voie) n'est pas occupée par un véhicule lent ; sinon il s'arrête *à la
ligne* (`BlockReason::KEEP_CLEAR`, label *DEGAGE*). (b) **Bris de cycle** par
ordre total (VIN) dans le filet anti-collision : dans tout cycle de conflits, le
plus petit identifiant ne cède jamais → il avance et débloque la rotation.

> **À INTÉGRER :** capture du gridlock initial (`...135834.png`) puis du même
> carrefour fluidifié après correction.

### 5.3. Dépassement — collisions frontales

**Correction.** Critère de sécurité MOBLEMENT explicite : TTC ≥ 4 s **et**
décélération requise de l'oncoming ≤ $b_{\text{safe}} = 2\,b_{\text{conf}}$
pour *démarrer* ; abort si l'oncoming redevient dangereux. De plus, pendant un
dépassement, un véhicule **frontal** (cap opposé >135°) déclenche toujours le
freinage (l'ordre par VIN ne s'applique pas à un frontal).

### 5.4. Franchissement de feu **rouge**

**Symptôme.** Un véhicule franchissait un feu **rouge** (pas même orange) alors
qu'il avait le temps de s'arrêter.

**Cause.** Un *commit* aveugle : tout véhicule à moins de ~25 px du carrefour
était « engagé » de force (pour ne pas s'arrêter au milieu), **sans consulter
l'état du feu** — l'exception ne couvrait que les STOP, pas les feux rouges.

**Correction (zone de dilemme).** Le commit n'est accordé que si (i) la voie est
libre (vert / gap accepté), ou (ii) le véhicule est **physiquement** sur l'aire
(`interOn`), ou (iii) il est dans la **zone de dilemme** — incapable de
s'arrêter avant la ligne *même en freinage d'urgence*
($v^2/2b_{\text{urg}} > $ distance à la ligne). Sinon il **respecte le rouge**,
quitte à freiner ferme. Plus aucun franchissement « alors qu'on avait le
temps ». Implémentation : `Vehicle::computeDecision`.

## 6. Méthodologie expérimentale

- **Échelle.** Scénario *Ville XXXXL* : réseau **irrégulier** 92×72 (blocs de
  tailles variées), ~50 carrefours tous modes confondus **+ 6 ronds-points de
  tailles différentes** (côté 4/6/8), ~500 véhicules actifs ; dézoom caméra
  étendu pour embrasser toute la ville.
- **Monte-Carlo headless.** `ExperimentRunner` balaye `stratégie × densité ×
  runs` sur un carrefour isolé, avec *warm-up* (transitoire ignoré) puis fenêtre
  de mesure. Hétérogénéité des conducteurs (profils + bruit gaussien) pour la
  robustesse.
- **Métriques** (cf. `MetricsCollector`) : débit (véh/min), délai moyen (temps
  perdu vs flux libre), vitesse moyenne, **TTC** min + violations (< 1,5 s),
  arrêts, jerk (confort).

### Comment produire vos CSV

```bash
# 1) Banc Monte-Carlo headless (perf + balayage) :
cmake -S . -B build -DTRAFFIC_BUILD_APP=OFF
cmake --build build -t simbench
./build/simbench 30 3                       # affiche les BENCH + résultats

# 2) Balayage exporté en CSV : via l'app (panneau « Monte-Carlo headless »
#    -> bouton Exporter CSV) -> enregistrez sous analysis/data/montecarlo.csv
#    (colonnes : strategy,density_veh_s,throughput_per_min,mean_delay_s,
#     mean_speed_px_s,min_ttc_s,ttc_violations,total_stops,completed)

# 3) Métriques d'une simulation live (ex. Ville XXXXL) : panneau « Métriques »
#    -> Exporter CSV -> analysis/data/live_xxxxl.csv
```

Déposez les fichiers dans `analysis/data/` puis ré-exécutez ce notebook : les
figures basculeront automatiquement des données synthétiques vers vos données
réelles.

In [ ]:
# Configuration & chargeurs robustes (synthétique si CSV absent)
import io, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (8, 5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 11})

CANDIDATE_DIRS = [Path("analysis/data"), Path("data"), Path(".")]

def _find(patterns):
    for pat in patterns:
        for d in CANDIDATE_DIRS:
            if not d.exists():
                continue
            hits = sorted(d.glob(pat))
            if hits:
                return hits[0]
    return None

SYNTHETIC_BANNER = ("⚠️  DONNÉES SYNTHÉTIQUES (illustratives) — "
                    "déposez vos CSV dans analysis/data/ pour les vraies figures.")
print("Répertoires de données scannés :", [str(d) for d in CANDIDATE_DIRS])

In [ ]:
# --- Chargement du balayage Monte-Carlo (ExperimentRunner) ---------------
MC_COLS = ["strategy", "density_veh_s", "throughput_per_min", "mean_delay_s",
           "mean_speed_px_s", "min_ttc_s", "ttc_violations", "total_stops",
           "completed"]

def load_montecarlo():
    p = _find(["montecarlo*.csv", "experiment*.csv", "mc_*.csv"])
    if p is not None:
        df = pd.read_csv(p)
        df.attrs["synthetic"] = False
        df.attrs["source"] = str(p)
        return df
    # --- Fallback synthétique : crossover P2P vs feux/priorité fixe ----------
    rng = np.random.default_rng(1337)
    dens = np.linspace(0.1, 0.8, 8)
    rows = []
    for strat, (cap, fixed_delay, decentral) in {
            "FIXED_PRIORITY": (26.0, 7.0, False),
            "P2P":            (24.0, 0.4, True)}.items():
        for d in dens:
            x = d / 0.45
            # P2P : délai quasi nul à basse densité, explose après le seuil.
            if decentral:
                delay = 0.4 + 22.0 * np.clip(x - 0.75, 0, None) ** 2
                thr = cap * d / (1 + 0.9 * np.clip(x - 0.8, 0, None) ** 2)
            else:
                delay = fixed_delay + 9.0 * d
                thr = cap * d / (1 + 0.15 * d)
            rows.append([strat, round(d, 3), round(thr, 2), round(delay, 2),
                         round(120 - 60 * d, 1),
                         round(max(0.6, 3.0 - 2.4 * d), 2),
                         int(40 * max(0, d - 0.5) ** 2 * (3 if decentral else 1)),
                         int(60 * d), int(thr * 0.9)])
    df = pd.DataFrame(rows, columns=MC_COLS)
    df.attrs["synthetic"] = True
    df.attrs["source"] = "synthetic"
    return df

mc = load_montecarlo()
if mc.attrs["synthetic"]:
    print(SYNTHETIC_BANNER)
else:
    print("Source Monte-Carlo :", mc.attrs["source"])
mc.head()

### 7.1. Débit (throughput) vs densité d'injection

In [ ]:
fig, ax = plt.subplots()
for strat, g in mc.groupby("strategy"):
    g = g.sort_values("density_veh_s")
    ax.plot(g["density_veh_s"], g["throughput_per_min"], "-o", label=strat)
ax.set_xlabel("Densité d'injection (véh/s)")
ax.set_ylabel("Débit évacué (véh/min)")
ax.set_title("Débit vs densité" + ("  [synthétique]" if mc.attrs["synthetic"] else ""))
ax.legend()
plt.tight_layout(); plt.show()

**Lecture.** À faible densité, la négociation décentralisée (P2P) maintient
un débit proche de l'idéal (peu d'arrêts). Au-delà d'un **seuil critique**, les
conflits se multiplient : le P2P se comporte comme un carrefour « 4 STOP » et son
débit plafonne puis chute — alors que feux/priorité fixe, bornés mais stables,
finissent par le dépasser (paradoxe de Braess).

### 7.2. Délai moyen vs densité

In [ ]:
fig, ax = plt.subplots()
for strat, g in mc.groupby("strategy"):
    g = g.sort_values("density_veh_s")
    ax.plot(g["density_veh_s"], g["mean_delay_s"], "-o", label=strat)
ax.set_xlabel("Densité d'injection (véh/s)")
ax.set_ylabel("Délai moyen (s)")
ax.set_title("Délai vs densité" + ("  [synthétique]" if mc.attrs["synthetic"] else ""))
ax.legend()
plt.tight_layout(); plt.show()

> **À INTÉGRER :** commentez ici le point d'inflexion *observé sur vos
> données* (densité seuil où les courbes se croisent).

### 7.3. Sécurité — TTC minimal vs densité

In [ ]:
fig, ax = plt.subplots()
for strat, g in mc.groupby("strategy"):
    g = g.sort_values("density_veh_s")
    ax.plot(g["density_veh_s"], g["min_ttc_s"], "-o", label=strat)
ax.axhline(1.5, color="red", ls="--", lw=1, label="seuil critique 1,5 s")
ax.set_xlabel("Densité d'injection (véh/s)")
ax.set_ylabel("TTC minimal (s)")
ax.set_title("Sécurité (TTC) vs densité" + ("  [synthétique]" if mc.attrs["synthetic"] else ""))
ax.legend()
plt.tight_layout(); plt.show()

### 7.4. Performance moteur — multithreading & passage à l'échelle

`simbench` rapporte des lignes `BENCH key=value` (dont
`speedup_x_realtime`). Pour une comparaison FPS / ms-par-tick en fonction du
nombre d'agents (avec/sans threading), renseignez `analysis/data/perf.csv` au
format : `agents,ms_per_tick_single,ms_per_tick_threaded`.

In [ ]:
def load_perf():
    p = _find(["perf*.csv"])
    if p is not None:
        df = pd.read_csv(p); df.attrs["synthetic"] = False; return df
    agents = [100, 300, 500, 800, 1000]
    single = [1.2, 9.5, 26.0, 64.0, 100.0]     # O(N^2) : s'effondre
    threaded = [0.5, 2.3, 5.1, 9.8, 14.5]       # partition + multicœur
    df = pd.DataFrame({"agents": agents,
                       "ms_per_tick_single": single,
                       "ms_per_tick_threaded": threaded})
    df.attrs["synthetic"] = True
    return df

perf = load_perf()
x = np.arange(len(perf)); w = 0.38
fig, ax = plt.subplots()
ax.bar(x - w/2, perf["ms_per_tick_single"], w, label="monothread")
ax.bar(x + w/2, perf["ms_per_tick_threaded"], w, label="multithread + grille")
ax.axhline(1000/60, color="green", ls="--", lw=1, label="budget 60 FPS (16,7 ms)")
ax.set_xticks(x); ax.set_xticklabels(perf["agents"])
ax.set_xlabel("Nombre d'agents"); ax.set_ylabel("ms / tick")
ax.set_title("Coût par pas vs nombre d'agents" + ("  [synthétique]" if perf.attrs.get("synthetic") else ""))
ax.legend()
if perf.attrs.get("synthetic"): print(SYNTHETIC_BANNER)
plt.tight_layout(); plt.show()

**Lecture.** L'approche monothread s'effondre au-delà de quelques
centaines d'agents ($O(N^2)$) ; le partitionnement spatial multithreadé reste
sous le budget temps réel jusqu'à ~1000 agents (ou accélère drastiquement le
mode Monte-Carlo headless).

### 7.5. Séries temporelles d'une simulation live (Ville XXXXL)

Le CSV de `MetricsCollector` contient un résumé puis une section *séries
temporelles* (`sample,throughput_per_min,mean_delay_s,mean_speed,min_ttc`).

In [ ]:
def load_live_series():
    p = _find(["live*.csv", "metriques*.csv", "metrics*.csv"])
    if p is None:
        # Série synthétique : montée en charge puis régime stationnaire.
        t = np.arange(0, 240)
        thr = 18 * (1 - np.exp(-t/40)) + np.random.default_rng(1).normal(0, .6, t.size)
        df = pd.DataFrame({"sample": t, "throughput_per_min": thr,
                           "mean_delay_s": 3 + 5*(1-np.exp(-t/60)),
                           "mean_speed": 90 - 30*(1-np.exp(-t/50)),
                           "min_ttc": 2.2 + np.random.default_rng(2).normal(0,.2,t.size)})
        df.attrs["synthetic"] = True
        return df
    raw = p.read_text(encoding="utf-8", errors="ignore").splitlines()
    # Trouve l'en-tête de la section séries temporelles.
    hdr = next((i for i, l in enumerate(raw)
                if l.startswith("sample,")), None)
    if hdr is None:
        raise ValueError("section 'séries temporelles' introuvable dans " + str(p))
    df = pd.read_csv(io.StringIO("\n".join(raw[hdr:])))
    df.attrs["synthetic"] = False
    return df

live = load_live_series()
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col, lbl in zip(axes.ravel(),
        ["throughput_per_min", "mean_delay_s", "mean_speed", "min_ttc"],
        ["Débit (véh/min)", "Délai moyen (s)", "Vitesse moyenne (px/s)", "TTC min (s)"]):
    if col in live:
        ax.plot(live["sample"], live[col]); ax.set_title(lbl); ax.set_xlabel("échantillon")
if live.attrs.get("synthetic"): print(SYNTHETIC_BANNER)
plt.tight_layout(); plt.show()

## 8. Galerie — captures, GIFs & vidéos *(à intégrer)*

Déposez vos médias dans `analysis/figures/` et décommentez/complétez les
cellules ci-dessous. Suggestions de contenu :

1. **Ville XXXXL fluide** (vue dézoomée, ~1000 agents sans lag) — image ou GIF.
2. **Avant/après** du faux *SUIT* (§5.1) et du *gridlock* (§5.2).
3. **Feu rouge respecté** : un véhicule qui freine net à la ligne (§5.4).
4. **Dépassement** réussi vs avorté (oncoming) (§5.3).
5. **Peloton virtuel / CACC** : passage entrelacé sans arrêt.
6. **Mini-vidéo** (mp4) d'un carrefour P2P qui se dégrade en montant la densité.

In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown
FIG = Path("analysis/figures")
wanted = ["xxxxl_fluide.png", "suit_avant.png", "suit_apres.png",
          "gridlock_avant.png", "gridlock_apres.png", "feu_rouge.png"]
found = [f for f in wanted if (FIG / f).exists()] if FIG.exists() else []
if not found:
    display(Markdown("> *(aucune image dans `analysis/figures/` — déposez vos captures "
                     "en utilisant les noms suggérés ci-dessus pour qu'elles s'affichent ici.)*"))
for f in found:
    display(Markdown(f"**{f}**")); display(Image(filename=str(FIG / f)))

## 9. Conclusion & perspectives

Le passage d'un automate déterministe simple à un **système multi-agents
stochastique** fondé sur des modèles physiques (IDM/MOBIL, Frenet) transforme le
logiciel en un véritable **banc d'essai scientifique**. Les nuances observées —
notamment l'effondrement de l'efficacité P2P au-delà d'un seuil de densité —
confirment empiriquement le paradoxe de Braess et valident l'implémentation.

**Perspectives.** (i) *Virtual platooning* généralisé pour synchroniser les flux
inter-voies sans arrêt ; (ii) apprentissage des paramètres de politesse MOBIL ;
(iii) comparaison AIM centralisé vs P2P décentralisé sous panne de communication ;
(iv) calibration des unités px↔m pour des métriques en SI.

---
*Notebook généré par `analysis/_build_report.py`. Les figures se mettent à jour
automatiquement dès que les CSV réels sont déposés dans `analysis/data/`.*